In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torchaudio
from safetensors.torch import save_file
from model_audio_input.model import SpecTTTra
import torch
from model_audio_input.dataset import SonicDataset
from config.loader import get_training_model_kwargs
from safetensors.torch import load_file
import librosa

In [2]:
def load_model(
    weights_path: str,
    num_classes: int = 2,
    variant_name: str | None = None,
) -> nn.Module:
    """Load SpecTTTra with safetensors weights using config-driven model kwargs."""
    model_kwargs = get_training_model_kwargs(variant_name=variant_name)
    model_kwargs["num_classes"] = num_classes
    model_kwargs.setdefault("expected_samples", 96000)

    model = SpecTTTra(**model_kwargs)
    state_dict = load_file(weights_path)
    model.load_state_dict(state_dict)
    model.eval()
    return model

In [17]:
weight_path = "app/model_prob/model_alpha.safetensors"
model = load_model(weight_path, num_classes=1)
print("Loaded model")

Loaded model


In [5]:
audio_path = "hand_test_audio/50namAI.mp4"
waveform, sr = librosa.load(audio_path)   # [C, T]
waveform = torch.as_tensor(waveform, dtype=torch.float32)
if waveform.ndim == 1:
    waveform = waveform.unsqueeze(0)
elif waveform.ndim == 2 and waveform.shape[0] > waveform.shape[1]:
    waveform = waveform.transpose(0, 1)
sample_rate = 16000
expected_samples = 96000

# convert về mono nếu nhiều channel
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0)
else:
    waveform = waveform.squeeze(0)  # [T]

# resample nếu cần
if sr != sample_rate:
    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=sample_rate)
    waveform = resampler(waveform.unsqueeze(0)).squeeze(0)

# Model receives raw waveform and performs mel extraction inside forward().
waveform = waveform.to(torch.float32)
if waveform.numel() < expected_samples:
    pad = expected_samples - waveform.numel()
    waveform = torch.nn.functional.pad(waveform, (0, pad))
elif waveform.numel() > expected_samples:
    waveform = waveform[: expected_samples]


/tmp/ipykernel_66327/236576824.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  waveform, sr = librosa.load(audio_path)   # [C, T]


In [6]:
print(waveform.shape)
waveform = waveform.unsqueeze(0)   # [1, 96000]
print(waveform.shape)

torch.Size([96000])
torch.Size([1, 96000])


In [7]:
with torch.no_grad():
    logits = model(waveform)  # [1, 2]
print(logits)
prob = torch.sigmoid(logits)
pred = (prob >= 0.5).long()
print(prob)
print(pred)

tensor([[11.9167]])
tensor([[1.0000]])
tensor([[1]])


# Fully Detect

In [8]:
def detect_min_non_ai_chunk(
    audio_path: str,
    model,
    sample_rate: int = 16000,
    chunk_seconds: int = 6,
    batch_size: int = 32,
    non_ai_class_index: int = 1,  # nếu model 2-class và class 1 = human/non-AI
):
    model.eval()
    device = next(model.parameters()).device

    wav_np, sr = librosa.load(audio_path, sr=None, mono=True)
    wav = torch.tensor(wav_np, dtype=torch.float32)

    if sr != sample_rate:
        wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=sample_rate)

    chunk_size = sample_rate * chunk_seconds
    total_samples = wav.numel()

    chunks, starts = [], []
    for start in range(0, total_samples, chunk_size):
        chunk = wav[start:start + chunk_size]
        if chunk.numel() < chunk_size:
            chunk = torch.nn.functional.pad(chunk, (0, chunk_size - chunk.numel()))
        chunks.append(chunk)
        starts.append(start)

    x = torch.stack(chunks, dim=0)  # [N, 96000]

    non_ai_probs = []
    with torch.no_grad():
        for i in range(0, x.size(0), batch_size):
            xb = x[i:i + batch_size].to(device)
            logits = model(xb)

            # 1-logit: sigmoid cho p_non_ai
            if logits.ndim == 1 or logits.shape[-1] == 1:
                p_non_ai = torch.sigmoid(logits.view(-1))
            else:
                # 2-class: lấy xác suất class non-AI
                p_non_ai = torch.softmax(logits, dim=-1)[:, non_ai_class_index]

            non_ai_probs.append(p_non_ai.cpu())

    non_ai_probs = torch.cat(non_ai_probs, dim=0)  # [N]

    # Đếm chunk có p_non_ai < 0.5
    count_suspected_ai = int((non_ai_probs < 0.5).sum().item())

    # Chunk nghi AI nhất = p_non_ai thấp nhất
    best_idx = int(torch.argmin(non_ai_probs).item())
    best_p_non_ai = float(non_ai_probs[best_idx].item())
    best_p_ai = 1.0 - best_p_non_ai

    best_start_sample = starts[best_idx]
    best_end_sample = min(best_start_sample + chunk_size, total_samples)

    return {
        "audio_path": audio_path,
        "num_chunks": len(chunks),
        "count_chunks_non_ai_lt_50": count_suspected_ai,  # biến đếm bạn cần
        "best_chunk_index": best_idx,
        "best_non_ai_prob": best_p_non_ai,
        "best_ai_prob": best_p_ai,
        "best_start_sec": best_start_sample / sample_rate,
        "best_end_sec": best_end_sample / sample_rate,
        "all_non_ai_probs": non_ai_probs.tolist(),
    }


In [10]:
out = detect_min_non_ai_chunk("hand_test_audio/50namAI.mp4", model, sample_rate=16000, chunk_seconds=6, batch_size=16)
print(out)

/tmp/ipykernel_66327/912492504.py:12: UserWarning: PySoundFile failed. Trying audioread instead.
  wav_np, sr = librosa.load(audio_path, sr=None, mono=True)


{'audio_path': 'hand_test_audio/50namAI.mp4', 'num_chunks': 34, 'count_chunks_non_ai_lt_50': 1, 'best_chunk_index': 33, 'best_non_ai_prob': 0.03966403380036354, 'best_ai_prob': 0.9603359661996365, 'best_start_sec': 198.0, 'best_end_sec': 199.80775, 'all_non_ai_probs': [0.9999933242797852, 0.9999957084655762, 0.9999955892562866, 0.9999887943267822, 0.9999794960021973, 0.9999959468841553, 0.9999957084655762, 0.9999955892562866, 0.9999949932098389, 0.999995231628418, 0.9999954700469971, 0.9999953508377075, 0.9999954700469971, 0.9999948740005493, 0.9999951124191284, 0.9999953508377075, 0.999995231628418, 0.9999940395355225, 0.9999948740005493, 0.9999953508377075, 0.9999957084655762, 0.9999957084655762, 0.9999946355819702, 0.9999953508377075, 0.9999819993972778, 0.9999948740005493, 0.9999954700469971, 0.9999951124191284, 0.9999957084655762, 0.9999953508377075, 0.9999953508377075, 0.9999953508377075, 0.9999911785125732, 0.03966403380036354]}


In [11]:
out

{'audio_path': 'hand_test_audio/50namAI.mp4',
 'num_chunks': 34,
 'count_chunks_non_ai_lt_50': 1,
 'best_chunk_index': 33,
 'best_non_ai_prob': 0.03966403380036354,
 'best_ai_prob': 0.9603359661996365,
 'best_start_sec': 198.0,
 'best_end_sec': 199.80775,
 'all_non_ai_probs': [0.9999933242797852,
  0.9999957084655762,
  0.9999955892562866,
  0.9999887943267822,
  0.9999794960021973,
  0.9999959468841553,
  0.9999957084655762,
  0.9999955892562866,
  0.9999949932098389,
  0.999995231628418,
  0.9999954700469971,
  0.9999953508377075,
  0.9999954700469971,
  0.9999948740005493,
  0.9999951124191284,
  0.9999953508377075,
  0.999995231628418,
  0.9999940395355225,
  0.9999948740005493,
  0.9999953508377075,
  0.9999957084655762,
  0.9999957084655762,
  0.9999946355819702,
  0.9999953508377075,
  0.9999819993972778,
  0.9999948740005493,
  0.9999954700469971,
  0.9999951124191284,
  0.9999957084655762,
  0.9999953508377075,
  0.9999953508377075,
  0.9999953508377075,
  0.9999911785125732,
